# Conv2D V2 核函数开发

本章以 **Conv2D V2** 算子为例，完整演示如何在 AscendC 中直接调用卷积核函数。内容涵盖：

- 算子分析：卷积数学表达式、输入/输出规格、NPU 片上存储层次。
- 核函数开发：Tiling 结构体、片上缓冲区管理、im2col、Mmad、Fixpipe 等关键步骤。
- 主机侧调用：SetTilingData、CheckTilingSpace、ACL 内存管理与核函数启动。

本章学习大纲如下：

- 算子分析：明确 Conv2D 的数学表达式、输入输出规格及 NPU 存储层次。
- 核函数开发：逐步实现设备侧卷积核函数的各个模块。
- 主机侧调用代码：编写 Host 侧驱动程序，完成 Tiling 填充、内存管理与核函数启动。
- 编译与运行：使用 CMake + 毕昇编译器完成编译并执行。

---
## 1. 环境准备

正式开始学习之前，先要对 Jupyter 环境进行初始化。以下代码完成了初始化并将环境变量导入 Jupyter 环境，同时创建代码目录，保证能正常使用毕昇编译器完成算子的开发及编译。

In [ ]:
import os, subprocess
os.makedirs("Sources/01.03", exist_ok=True)
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n Environment initialization process completed successfully!")

---

## 2. 算子分析

### 2.1 数学表达式

二维卷积（Conv2D）的数学表达式为：

$$Y[n, c_o, h_o, w_o] = \sum_{c_i} \sum_{k_h} \sum_{k_w} X[n,\, c_i,\, h_o \cdot s_H + k_h \cdot d_H,\, w_o \cdot s_W + k_w \cdot d_W] \times W[c_o, c_i, k_h, k_w]$$

其中 $s_H, s_W$ 为步长，$d_H, d_W$ 为膨胀系数。

### 2.2 输入与输出

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>张量</strong></td>
  <td align="center"><strong>形状（NCHW）</strong></td>
  <td align="center" style="width: 180px"><strong>取值范围</strong></td>
</tr>
<tr>
  <td align="left">输入特征图 X </td>
  <td align="left">(BATCH, CI, HI, WI) </td>
  <td align="left">fp16</td>
</tr>
<tr>
  <td align="left">卷积核 W </td>
  <td align="left"> (CO, CI, KH, KW) </td>
  <td align="left">fp16</td>
</tr>
<tr>
  <td align="left">输出特征图 Y </td>
  <td align="left"> (BATCH, CO, HO, WO) </td>
  <td align="left">fp16</td>
</tr>
</table>
输出尺寸公式：

$$HO = \lfloor (HI + pad\_top + pad\_bottom - d_H(KH-1) - 1) / s_H \rfloor + 1$$

### 2.3 NPU 上存储层次

Conv2D 的数据流经过以下存储层次（速度从快到慢）：

```
GM  (Global Memory)  — 全局存储空间 (Ascend 950PR/Ascend 950DT)
L1  (L1 缓冲区)             — 每核 512 MB
L0A (矩阵 A 缓冲区)         — 64 KB，送入 Mmad 左操作数
L0B (矩阵 B 缓冲区)         — 64 KB，送入 Mmad 右操作数
L0C (累加器缓冲区)           — 256 KB，存储 Mmad 结果（fp32）
```

数据流示意：

```
GM ──DataCopy──► L1  (LoadAL1 / LoadBL1)
L1 ──Load3D──►  L0A  (LoadAL0: im2col 在线展开)
L1 ──Load2D──►  L0B  (LoadBL0: 权重 NZ 布局)
L0A × L0B ──Mmad──► L0C
L0C ──Fixpipe──► GM  (fp32→fp16 类型转换 + NCHW 布局)
```

### 2.4 本 Demo 的局限性与约束

本 Demo 是一个**最小可运行实现**，**只支持Ascend 950PR/Ascend 950DT芯片型号**，目的是清晰展示 AscendC 卷积核函数的完整调用链。为了降低复杂度，做了以下简化，在实际生产代码中均需扩展：

#### 全载搬运（Full-Load）

这是最核心的限制。`SetTilingData` 中令 `kL0 == kAL1 == kBL1 == KH × KW × CI`，即整个 K 维度在一次 L1→L0 搬运中全部完成，`ddr2l0LoopK` 始终为 1，K 循环只跑一次。

当输入通道 CI 或卷积核尺寸 KH/KW 增大时，全载所需的 L0A/L0B 空间会线性增长，很快超出 64 KB 的硬件上限。生产实现需要将 K 维度分块，分多次 Mmad 累加。

#### 单核，无多核分工

`numBlocks = 1`，整个输出特征图由单核完成。没有按 batch、输出通道或空间维度分核的逻辑（无 `block_idx` 分片）。

#### 无流水 / 无双缓冲

L1 队列深度为 1（`TQue<..., 1>`），L0 双缓冲关闭（`L0_SYBC_DB_CLOSE`）。数据搬运（MTE1）和矩阵计算（M）串行执行，没有流水重叠，硬件利用率低。

#### 数据类型固定为 fp16

输入特征图、权重、输出均为 `half`（fp16），L0C 累加器为 `float`（fp32）。不支持 int8、bf16 等其他数据类型。

#### 数据布局固定为 NCHW

`ConvFormat` 枚举只定义了 `NCHW = 1`，不支持 NHWC 等其他布局。

#### CI 和 CO 均固定为 16

- **CI = 16**：`aL1SpaceSize = 8192` 硬编码，只够容纳 CI=16 的一个输入 patch；`cinAInCore / cinBInCore` 也直接由 `kAL1 / kernelHW` 推算，没有多 CI 块的循环。
- **CO = 16**：`nBL1 = 16`、`nL0 = 16` 硬编码，每次只处理 16 个输出通道，没有多 CO 块的循环。

两者的根本原因相同：Demo 没有实现通道方向的分块循环，CI 和 CO 必须恰好等于 16（即一个 C0 块），而不是"16 的倍数"。

#### 只支持对称 Padding

`Conv2DTilingData` 只存储 `padTop` 和 `padLeft`，没有独立的 `padBottom` 和 `padRight` 字段。`LoadAL1` 在运行时从输入/输出尺寸的几何关系推算 bottom/right padding，隐含 `padBottom = padTop`、`padRight = padLeft`。非对称 padding 无法通过当前结构体表达。

#### 空间尺寸上限（由全载搬运决定）

`CheckTilingSpace` 会在启动前验证以下约束，超出则拒绝运行：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>缓冲区</strong></td>
  <td align="center"><strong>硬件上限</strong></td>
  <td align="center" style="width: 300px"><strong>全载用量公式</strong></td>
</tr>
<tr>
  <td align="left">L1 </td>
  <td align="left">512 MB / 核</td>
  <td align="left">`(hiLoad × wiLoad × CI + nBL1 × kBL1) × 2` 字节</td>
</tr>
<tr>
  <td align="left">L0A </td>
  <td align="left"> 64 KB / 核 </td>
  <td align="left"> `AlignUp(HO×WO, 16) × (KH×KW×CI) × 2` 字节 </td>
</tr>
<tr>
  <td align="left">L0B </td>
  <td align="left">  64 KB / 核  </td>
  <td align="left"> `AlignUp(CO, 16) × (KH×KW×CI) × 2` 字节</td>
</tr>
<tr>
  <td align="left">L0C </td>
  <td align="left">  256 KB / 核  </td>
  <td align="left"> `AlignUp(HO×WO, 16) × AlignUp(CO, 16) × 4` 字节 </td>
</tr>
</table>
其中 `hiLoad = (HO-1)×strideH + dilatedKH`，`wiLoad = (WO-1)×strideW + dilatedKW`。

> **示例验证**：本 Demo 默认参数（CI=16, HI=WI=8, KH=KW=3, CO=16, 无 padding）下，输入全 1、权重全 1，每个输出点的计算值为：
>
> $$Y[n, c_o, h_o, w_o] = \sum_{c_i=0}^{15} \sum_{k_h=0}^{2} \sum_{k_w=0}^{2} 1 \times 1 = CI \times KH \times KW = 16 \times 3 \times 3 = 144$$
>
> 输出通道 $c_o$ 不影响每个点的数值，因为所有输出通道的权重均为 1，累加的元素数量（CI × KH × KW）相同。

---

## 3. 核函数开发

基于对 Conv2D V2 算子的分析，我们开始逐步实现核函数代码。所有代码均写入 `Sources/01.03/minimal_demo.asc`，采用增量追加方式，便于逐节理解每个模块的作用。

### 3.1 头文件与 Tiling 结构体

首先引入必要的头文件，并定义 `Conv2DTilingData` 结构体。

**Conv2DTilingData** 保存了核函数所需的全部参数：存储层次的分块尺寸以及 padding 偏移量。它由主机侧通过 `aclrtMemcpy` 传递给设备侧，再由 `CopyConv2dTiling` 从 GM 复制到片上内存后供核函数读取。

**CopyConv2dTiling / GET_TILING_DATA**：核函数不能直接解引用 GM 指针读取结构体字段，必须先将数据复制到片上（L1 可访问）内存。`CopyConv2dTiling` 逐字完成这一复制，`GET_TILING_DATA` 宏将其封装为一行调用。

In [ ]:
%%writefile Sources/01.03/minimal_demo.asc
/**
 * Copyright (c) 2025 Huawei Technologies Co., Ltd.
 */

/*!
 * \file conv2d_v2_demo.asc
 * \brief Standalone demo for direct-calling Conv2D V2 kernel.
 *
 * ============================================================
 * Tutorial overview
 * ============================================================
 * This file is the host-side driver for the Conv2D V2 kernel demo.
 * It covers the full lifecycle of an AscendC kernel call:
 *
 *   1. Define conv shape constants (BATCH, CI, HI, WI, CO, KH, KW, …)
 *   2. Fill in the tiling struct (SetTilingData) — tells the kernel how
 *      to partition work across the NPU memory hierarchy.
 *   3. Validate that the tiling fits in on-chip SRAM (CheckTilingSpace).
 *   4. Allocate device memory, copy inputs, launch the kernel.
 *   5. Copy results back and dump to text files for inspection.
 *
 * NPU memory hierarchy (smallest / fastest → largest / slowest):
 *
 *   GM  (Global Memory)  — off-chip (Ascend 950PR/Ascend 950DT)
 *   L1  (on-chip SRAM)         — 512 MB per core
 *   L0A (Matrix A buffer)      — 64 KB, feeds left operand of Mmad
 *   L0B (Matrix B buffer)      — 64 KB, feeds right operand of Mmad
 *   L0C (Accumulator buffer)   — 256 KB, stores Mmad result (fp32)
 *
 * Data flow for a single Conv2D tile:
 *
 *   GM ──DataCopy──► L1  (LoadAL1 / LoadBL1)
 *   L1 ──Load3D──►  L0A  (LoadAL0: im2col on-the-fly)
 *   L1 ──Load2D──►  L0B  (LoadBL0: weight NZ layout)
 *   L0A × L0B ──Mmad──► L0C
 *   L0C ──Fixpipe──► GM  (fp32→fp16 cast + NCHW layout)
 * ============================================================
 */

#include <cstdint>
#include <cstring>
#include <iostream>
#include <fstream>
#include <cmath>
#include "acl/acl.h"
#include "kernel_operator.h"

// ============================================================================
// Tiling data structure
// ============================================================================
//
// Conv2DTilingData holds all parameters the kernel needs: memory-hierarchy
//   tiling values plus the two padding offsets.
//   Passed from host to device via aclrtMemcpy, then copied from GM to
//   on-chip memory by CopyConv2dTiling before the kernel reads any field.
//
// CopyConv2dTiling / GET_TILING_DATA: the kernel cannot dereference a GM
//   pointer directly to read struct fields — the data must first be copied
//   to on-chip (L1-accessible) memory.  CopyConv2dTiling does that word-by-word.

#pragma pack(1)

struct Conv2DTilingData {
    uint64_t orgHi = 0;
    uint64_t orgWi = 0;
    uint64_t orgHo = 0;
    uint64_t orgWo = 0;
    uint64_t singleCoreBatch = 0;
    uint64_t singleCoreHo = 0;
    uint64_t singleCoreWo = 0;
    uint32_t orgCi = 0;
    uint32_t orgCo = 0;
    uint32_t singleCoreCi = 0;
    uint32_t singleCoreCo = 0;
    uint32_t kAL1 = 0;
    uint32_t kBL1 = 0;
    uint32_t nBL1 = 0;
    uint32_t hoL0 = 0;
    uint32_t woL0 = 0;
    uint32_t kL0 = 0;
    uint32_t nL0 = 0;
    uint32_t orgHixWi = 0;
    uint32_t kernelHxkernelW = 0;
    uint32_t aL1SpaceSize = 0;
    uint32_t cinAInCore = 0;
    uint32_t cinBInCore = 0;
    uint32_t mStep = 0;
    uint32_t kStep = 0;
    uint32_t fmapKStride = 0;
    uint32_t coutOffsetBlock = 0;
    uint32_t nL1DivBlockSize = 0;
    uint32_t kernelH = 0;
    uint32_t kernelW = 0;
    uint32_t strideH = 0;
    uint32_t strideW = 0;
    uint32_t dilationH = 0;
    uint32_t dilationW = 0;
    int8_t offsetx = 0;
    uint32_t padTop = 0;
    uint32_t padLeft = 0;
};

#pragma pack()

__aicore__ inline void CopyConv2dTiling(Conv2DTilingData* dst, GM_ADDR tilingGM)
{
    uint32_t* ptr = reinterpret_cast<uint32_t*>(dst);
    auto src = reinterpret_cast<__gm__ uint32_t*>(tilingGM);
    for (uint32_t i = 0; i < sizeof(Conv2DTilingData) / sizeof(uint32_t); i++, ptr++) {
        *ptr = *(src + i);
    }
}

#define GET_TILING_DATA(tilingData, tilingArg) \
    Conv2DTilingData tilingData;               \
    CopyConv2dTiling(&tilingData, tilingArg)



### 3.2 NPU存储层次与常量

`#ifdef __CCE_AICORE__` 块内的代码只在设备侧编译。`namespace conv` 中定义了各个缓冲区的大小常量（L0A/L0B/L0C 均以字节为单位）、C0 块大小（32 字节）、padding 索引以及 Load3D/Load2D 所需的位域偏移量。

这些常量直接对应 NPU 硬件规格，修改它们会导致越界访问，因此在教学中保持固定值。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
#ifdef __CCE_AICORE__
#include "kernel_basic_intf.h"
#include "kernel_tiling/kernel_tiling.h"

using namespace AscendC;

namespace conv {

const static uint64_t L0A_SIZE = 65536;
const static uint64_t L0B_SIZE = 65536;
const static uint64_t L0C_SIZE = 262144;
const static uint64_t C0_SIZE = 32;
const static uint64_t PAD_IDX_T = 2;
const static uint64_t PAD_IDX_L = 0;
const static uint64_t PAD_IDX_R = 1;
const static uint64_t MAX_PAD_R = 255;
const static uint64_t MIN_HI_WI = 1;
const static uint32_t BLOCK_L0_N = 16;
const static uint32_t BLOCK_L0_M = 16;
const static uint32_t L0_SYBC_DB_CLOSE = 0x0;

const static uint8_t MSTEP_OFFSET = 16;
const static uint8_t POSM_OFFSET = 48;
const static uint8_t POSK_OFFSET = 32;
const static uint8_t STRIDEH_OFFSET = 6;
const static uint8_t KERNELW_OFFSET = 12;
const static uint8_t KERNELH_OFFSET = 20;
const static uint8_t KERNELW_HIGHEST_BIT_OFFSET = 36;
const static uint8_t KERNELH_HIGHEST_BIT_OFFSET = 37;
const static uint8_t DILATIONW_OFFSET = 28;
const static uint8_t DILATIONH_OFFSET = 36;
const static uint8_t CIN_OFFSET = 48;

const static uint64_t MASK_16 = 0xffff;
const static uint64_t MASK_8 = 0xff;
const static uint64_t MASK_6 = 0x3f;
const static uint64_t NINTH_BIT_MASK = 0x100;

const static uint64_t DEQ_SCALAR_ONE = 1065353216;
static constexpr IsResetLoad3dConfig CONV_LOAD3DV2_DEFAULT_CONFIG = {false, false};

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 5102)
#define ASCEND_IS_AIC_CONV constexpr(true)
#define ASCEND_IS_AIV_CONV constexpr(true)
#else
#define ASCEND_IS_AIC_CONV ASCEND_IS_AIC
#define ASCEND_IS_AIV_CONV ASCEND_IS_AIV
#endif

constexpr uint8_t UNIT_FLAG_ENABLE_ONLY = 2;
constexpr uint8_t UNIT_FLAG_ENABLE_WITH_FLIP = 3;

static __aicore__ inline uint64_t AlignB(uint64_t a, uint64_t b)
{
    return ((a + b - 1) / b) * b;
}

static __aicore__ inline uint64_t CeilDiv(uint64_t a, uint64_t b)
{
    return (a + b - 1) / b;
}



### 3.3 数据类型定义

本节定义三个核心类型：

- **ConvFormat**：枚举卷积数据布局，当前仅支持 NCHW。
- **ConvType**：将存储位置（TPosition）、数据布局（ConvFormat）和元素类型（TYPE）打包为一个模板类型标签，供 `Conv2dContext` 和 `Conv2dBase` 使用。
- **Conv2dContext**：保存单次卷积迭代所需的全部状态，包括片上缓冲区句柄（al0Buf/bl0Buf/l0cBuf）、队列（queueAL1/queueBL1）、GlobalTensor 指针以及从 Tiling 派生的循环控制变量。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
enum class ConvFormat : std::uint8_t {
    NCHW = 1,
};

template <TPosition POSITION, ConvFormat FORMAT, typename TYPE>
struct ConvType {
    constexpr static TPosition pos = POSITION;
    constexpr static ConvFormat format = FORMAT;
    using T = TYPE;
};

template <class FMAP_TYPE, class WEIGHT_TYPE, class OUTPUT_TYPE>
struct Conv2dContext {
    using FmapT = typename FMAP_TYPE::T;
    using WeightT = typename WEIGHT_TYPE::T;
    using OutputT = typename OUTPUT_TYPE::T;
    using L0cT = float;

    constexpr static uint64_t k0 = C0_SIZE / sizeof(WeightT);

    TPipe pipe;
    GlobalTensor<FmapT> agm;
    GlobalTensor<WeightT> bgm;
    TBuf<TPosition::A2> al0Buf;
    TBuf<TPosition::B2> bl0Buf;
    TBuf<TPosition::CO1> l0cBuf;

    LocalTensor<FmapT> al1;
    LocalTensor<WeightT> bl1;
    LocalTensor<FmapT> wholeAl0Tensor;
    LocalTensor<FmapT> al0;
    LocalTensor<WeightT> wholeBl0Tensor;
    LocalTensor<WeightT> bl0;
    LocalTensor<L0cT> wholeCl0Tensor = LocalTensor<L0cT>(TPosition::CO1, 0, 0);
    LocalTensor<L0cT> cl0 = LocalTensor<L0cT>(TPosition::CO1, 0, 0);

    TQue<QuePosition::A1, 1> queueAL1;
    TQue<QuePosition::B1, 1> queueBL1;

    const Conv2DTilingData* __restrict convTiling = nullptr;

    uint64_t fmapOneBatchSize = 0;
    uint64_t outputOneBatchSize = 0;
    uint64_t dilatedKernelH = 0;
    uint64_t dilatedKernelW = 0;
    uint64_t orgCi = 0;
    uint64_t orgCo = 0;
    uint64_t orgHi = 0;
    uint64_t orgWi = 0;
    uint64_t orgHo = 0;
    uint64_t orgWo = 0;
    uint64_t kernelH = 0;
    uint64_t kernelW = 0;
    int64_t hiStartPos = -1;
    int64_t wiStartPos = 0;
    uint64_t singleCoreBatch = 0;
    uint64_t singleCoreHo = 0;
    uint64_t singleCoreWo = 0;
    uint64_t singleCoreCi = 0;
    uint64_t singleCoreCo = 0;

    uint64_t currentHoL0 = 0;
    uint64_t currentWoL0 = 0;
    uint64_t currentNBL1 = 0;
    uint64_t currentML0Align = 0;
    uint64_t currentNL0Align = 0;

    uint64_t ddr2l0LoopK = 0;
    uint64_t maxKL0Iter = 0;
    uint64_t kL0Tail = 0;
    uint64_t kIter = 0;
    uint64_t kAL0Iter = 0;
    uint64_t kBL0Iter = 0;

    uint32_t bL1SpaceSize = 0;
};



### 3.4 上下文初始化 InitContext

`InitContext` 从 Tiling 结构体中读取所有形状参数，初始化片上缓冲区（L0A/L0B/L0C）和 L1 队列，并预计算 K 维度的循环次数（`ddr2l0LoopK`）及尾块大小（`kL0Tail`）。

关键逻辑：
- `dilatedKernelH/W`：考虑膨胀后的有效卷积核尺寸。
- `ddr2l0LoopK = CeilDiv(totalK, kL0)`：K 维度需要多少次 L0 迭代。
- `kL0Tail`：最后一次迭代的 K 块大小（若整除则等于 kL0）。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
template <class Ctx>
__aicore__ inline void InitContext(Ctx &ctx, const Conv2DTilingData *tiling)
{
    ctx.convTiling = tiling;
    ctx.singleCoreBatch = tiling->singleCoreBatch;

    if ASCEND_IS_AIC_CONV {
        ctx.orgCi = tiling->orgCi;
        ctx.orgHi = tiling->orgHi;
        ctx.orgWi = tiling->orgWi;
        ctx.orgCo = tiling->orgCo;
        ctx.orgHo = tiling->orgHo;
        ctx.orgWo = tiling->orgWo;
        ctx.kernelH = tiling->kernelH;
        ctx.kernelW = tiling->kernelW;
        ctx.dilatedKernelH = 1 + (ctx.kernelH - 1) * tiling->dilationH;
        ctx.dilatedKernelW = 1 + (ctx.kernelW - 1) * tiling->dilationW;
        ctx.singleCoreCi = tiling->singleCoreCi;
        ctx.singleCoreCo = tiling->singleCoreCo;
        ctx.fmapOneBatchSize = tiling->orgCi * tiling->orgHi * tiling->orgWi;
        ctx.outputOneBatchSize = tiling->orgCo * tiling->orgHo * tiling->orgWo;

        ctx.singleCoreHo = tiling->singleCoreHo;
        ctx.singleCoreWo = tiling->singleCoreWo;
        ctx.currentHoL0 = ctx.singleCoreHo;
        ctx.currentWoL0 = ctx.singleCoreWo;
        ctx.currentNBL1 = tiling->nBL1;

        ctx.pipe.InitBuffer(ctx.al0Buf, L0A_SIZE);
        ctx.pipe.InitBuffer(ctx.bl0Buf, L0B_SIZE);
        ctx.pipe.InitBuffer(ctx.l0cBuf, L0C_SIZE);
        ctx.wholeAl0Tensor = ctx.al0Buf.template Get<typename Ctx::FmapT>();
        ctx.wholeBl0Tensor = ctx.bl0Buf.template Get<typename Ctx::WeightT>();
        ctx.wholeCl0Tensor = ctx.l0cBuf.template Get<typename Ctx::L0cT>();
        ctx.bL1SpaceSize = tiling->nBL1 * tiling->kBL1;
        ctx.pipe.InitBuffer(ctx.queueAL1, 1, tiling->aL1SpaceSize);
        ctx.pipe.InitBuffer(ctx.queueBL1, 1, ctx.bL1SpaceSize * sizeof(typename Ctx::WeightT));

        uint64_t totalK = AlignB(ctx.singleCoreCi, Ctx::k0) * tiling->kernelHxkernelW;
        ctx.ddr2l0LoopK = CeilDiv(totalK, tiling->kL0);
        ctx.maxKL0Iter = ctx.ddr2l0LoopK - 1;
        ctx.kL0Tail = totalK % tiling->kL0;
        ctx.kL0Tail = ctx.kL0Tail == 0 ? tiling->kL0 : ctx.kL0Tail;

        ctx.currentML0Align = tiling->mStep;
        ctx.currentNL0Align = tiling->nL0;
    }
}



### 3.5 数据加载：LoadAL1 / LoadBL1

**LoadAL1**：将特征图的一个输入 patch 从 GM 搬运到 L1（`queueAL1`）。

核心步骤：
1. 根据当前输出 tile 的位置（`hiStartPos`/`wiStartPos`）计算需要加载的输入行列范围，并推导出四个方向的 padding 量。
2. 若整个 patch 都落在 padding 区域（`allPad`），则用 `InitConstValue` 填零，避免越界访问。
3. 否则，使用 `Dn2NzParams` 描述 NZ 布局的搬运参数，调用 `DataCopy` 完成实际数据搬运。

**LoadBL1**：将权重从 GM 搬运到 L1（`queueBL1`）。

- 1×1 卷积（`kernelHxkernelW == 1`）使用 `Nd2NzParams`（行优先到 NZ 的转换）。
- 非 1×1 卷积使用 `Dn2NzParams`，按 `coutOffsetBlock` 步长遍历输出通道块。

#### DataCopy详解

**功能定位**：DataCopy是通用数据搬运接口，支持多种数据搬运场景，并可在搬运过程中实现随路格式转换和量化激活等操作。该接口支持Local Memory与Global Memory之间的数据搬运，以及Local Memory内部的数据搬运，不涉及img2col变换。

**核心参数**：

```cpp

DataCopyParams params;
params.blockCount = blockNum;       // 搬运块数
params.blockLen = dataLength;       // 每块数据长度
params.srcStride = srcStride;       // 源步长
params.dstStride = dstStride;       // 目的步长
// 带Padding的搬运
DataCopyPadParams padParams;
padParams.isPad = true;
padParams.leftPadding = leftPad;
padParams.rightPadding = rightPad;

```

**代码示例**：

```cpp

DataCopyParams dataCopyParams;
dataCopyParams.blockCount = cin1LoadL1;
dataCopyParams.blockLen = hiLoadL1 * orgWi;
dataCopyParams.srcStride = orgHixWi - blockLen;
DataCopy<FmapT>(al1[offset], agm[gmOffset], dataCopyParams);
// 带Padding搬运
DataCopyPadParams padParams(true, 0, rightPadding, 0);
DataCopyPad<ChannelWiseT>(tensorL1, tensorGm[offset], dataCopyParams, padParams);

```

**格式转换的必要性**：

- **硬件适配**：Cube单元要求输入为分形矩阵格式（如NZ格式），而非NCHW/NHWC的连续存储格式
- **性能优化**：分形格式能够更好地利用矩阵乘法单元的计算能力，提高内存访问效率
- **计算流水**：格式转换与数据搬运融合，减少额外的转换开销

**常用格式转换参数结构体**：
在卷积算子实现中，DataCopy不仅完成数据搬运，还承担着格式转换的关键角色。昇腾AI处理器的Cube单元要求输入数据为特定的分形格式（Fractal Format），而主流框架（如PyTorch、TensorFlow）通常采用NCHW或NHWC格式。因此，需要在数据从GM搬运至L1的过程中完成格式转换，以适配硬件计算单元的输入要求。为支持不同格式的数据转换，Ascend C提供了专门的参数结构体。以下介绍两种核心格式转换参数：Dn2NzParams和Nd2NzParams。

#### Dn2NzParams详解

**功能**：将NCHW格式转换为NZ格式。

**Dn2NzParams结构体参数定义**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>参数名称</strong></td>
  <td align="center"><strong>含义</strong></td>
  <td align="center" style="width: 180px"><strong>取值范围</strong></td>
</tr>
<tr>
  <td align="left">dnNum</td>
  <td align="left">传输DN矩阵的数目。</td>
  <td align="left">[0, 4095]</td>
</tr>
<tr>
  <td align="left">nValue</td>
  <td align="left">DN矩阵的行数（矩阵高度）。</td>
  <td align="left">[0, 16384]</td>
</tr>
<tr>
  <td align="left">dValue</td>
  <td align="left">DN矩阵的列数（矩阵宽度）。当dValue不满足32B对齐时，目的操作数中不足部分会被补齐为0。</td>
  <td align="left">[0, 2^32-1]</td>
</tr>
<tr>
  <td align="left">srcDnMatrixStride</td>
  <td align="left">源操作数相邻DN矩阵起始地址间的偏移，单位：元素。</td>
  <td align="left">[0, 2^64-1]</td>
</tr>
<tr>
  <td align="left">srcDValue</td>
  <td align="left">源操作数同一DN矩阵的相邻行起始地址间的偏移，单位：元素。</td>
  <td align="left">[1, 2^64-1]</td>
</tr>
<tr>
  <td align="left">dstNzC0Stride</td>
  <td align="left">DN转换到NZ格式后，源操作数中的一列会转换为目的操作数的多行。dstNzC0Stride表示目的NZ矩阵中，来自源操作数同一列的多行数据相邻行起始地址间的偏移，单位：C0_SIZE（32B）。</td>
  <td align="left">[1, 65535]</td>
</tr>
<tr>
  <td align="left">dstNzNStride</td>
  <td align="left">目的NZ矩阵中，Z型矩阵相邻行起始地址之间的偏移，单位：C0_SIZE（32B）。</td>
  <td align="left">[1, 65535]</td>
</tr>
<tr>
  <td align="left">dstNzMatrixStride</td>
  <td align="left">目的NZ矩阵中，相邻NZ矩阵起始地址间的偏移，单位：元素。</td>
  <td align="left">[1, 2^32-1]</td>
</tr>
</table>

#### Nd2NzParams详解

**功能**：将NHWC格式转换为NZ格式。

**Nd2NzParams结构体参数定义**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>参数名称</strong></td>
  <td align="center"><strong>含义</strong></td>
  <td align="center" style="width: 180px"><strong>取值范围</strong></td>
</tr>
<tr>
  <td align="left">ndNum</td>
  <td align="left">传输ND矩阵的数目。</td>
  <td align="left">[0, 4095]</td>
</tr>
<tr>
  <td align="left">nValue</td>
  <td align="left">ND矩阵的行数（矩阵高度）。</td>
  <td align="left">[0, 16384]</td>
</tr>
<tr>
  <td align="left">dValue</td>
  <td align="left">ND矩阵的列数（矩阵宽度）。当dValue不满足32B对齐时，目的操作数中不足部分会被补齐为0。</td>
  <td align="left">[0, 65535]</td>
</tr>
<tr>
  <td align="left">srcNdMatrixStride</td>
  <td align="left">源操作数相邻ND矩阵起始地址间的偏移，单位：元素。</td>
  <td align="left">[0, 65535]</td>
</tr>
<tr>
  <td align="left">srcDValue</td>
  <td align="left">源操作数同一ND矩阵的相邻行起始地址间的偏移，单位：元素。</td>
  <td align="left">[1, 65535]</td>
</tr>
<tr>
  <td align="left">dstNzC0Stride</td>
  <td align="left">ND转换到NZ格式后，源操作数中的一行会转换为目的操作数的多行。dstNzC0Stride表示目的NZ矩阵中，来自源操作数同一行的多行数据相邻行起始地址间的偏移，单位：C0_SIZE（32B）。</td>
  <td align="left">[1, 16384]</td>
</tr>
<tr>
  <td align="left">dstNzNStride</td>
  <td align="left">目的NZ矩阵中，Z型矩阵相邻行起始地址之间的偏移，单位：C0_SIZE（32B）。</td>
  <td align="left">[1, 16384]</td>
</tr>
<tr>
  <td align="left">dstNzMatrixStride</td>
  <td align="left">目的NZ矩阵中，相邻NZ矩阵起始地址间的偏移，单位：元素。</td>
  <td align="left">[1, 65535]</td>
</tr>
</table>

详细的ND2NZ转换示意图请参考[昇腾社区官方文档](https://www.hiascend.com/document/detail/zh/canncommercial/850/API/ascendcopapi/atlasascendc_api_07_00127.html)中的"图1 ND2NZ转换示意图（half数据类型）"部分。该图清晰地展示了从ND格式到NZ格式的转换过程，包括源操作数和目的操作数的数据布局、各参数对应的偏移关系以及DataBlock的对齐方式。

**约束说明**：

- 针对Atlas 推理系列产品AI Core，使用Global Memory → Local Memory通路的ND2NZ搬运接口时，需要预留8K的UB空间，作为接口的临时数据存放区。

#### 格式转换对比

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>参数类型</strong></td>
  <td align="center"><strong>源格式</strong></td>
  <td align="center"><strong>目标格式</strong></td>
  <td align="center"><strong>适用框架</strong></td>
</tr>
<tr>
  <td align="left">Dn2NzParams</td>
  <td align="left">NCHW</td>
  <td align="left">NZ</td>
  <td align="left">PyTorch、ONNX、Caffe</td>
</tr>
<tr>
  <td align="left">Nd2NzParams</td>
  <td align="left">NHWC</td>
  <td align="left">NZ</td>
  <td align="left">TensorFlow、推理引擎</td>
</tr>
</table>



In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
template <class Ctx>
__aicore__ inline void LoadAL1(Ctx &ctx)
{
    ctx.al1 = ctx.queueAL1.template AllocTensor<typename Ctx::FmapT>();

    const auto *t = ctx.convTiling;
    int64_t paddingHi = (ctx.singleCoreHo - 1) * t->strideH + ctx.dilatedKernelH;
    int64_t paddingWi = (ctx.singleCoreWo - 1) * t->strideW + ctx.dilatedKernelW;

    int64_t hiTop = ctx.hiStartPos > 0 ? 0 : ctx.hiStartPos;
    int64_t wiLeft = ctx.wiStartPos > 0 ? 0 : ctx.wiStartPos;
    int64_t hiBottom = hiTop + paddingHi;
    int64_t wiRight = wiLeft + paddingWi;

    uint64_t padTopL1 = hiTop < 0 ? uint64_t(-hiTop) : 0;
    hiBottom = ctx.hiStartPos > 0 ? hiBottom + ctx.hiStartPos : hiBottom;
    uint64_t padBottomL1 = hiBottom > (int64_t)ctx.orgHi ? uint64_t(hiBottom - ctx.orgHi) : 0;
    uint64_t padLeftL1 = wiLeft < 0 ? uint64_t(-wiLeft) : 0;
    wiRight = ctx.wiStartPos > 0 ? wiRight + ctx.wiStartPos : wiRight;
    uint64_t padRightL1 = wiRight > (int64_t)ctx.orgWi ? uint64_t(wiRight - ctx.orgWi) : 0;

    bool allPad = (padTopL1 >= (uint64_t)paddingHi || padBottomL1 >= (uint64_t)paddingHi ||
                   padLeftL1 >= (uint64_t)paddingWi || padRightL1 >= (uint64_t)paddingWi);
    uint64_t hiLoad = 0, wiLoad = 0;
    if (!allPad) {
        hiLoad = paddingHi - padTopL1 - padBottomL1;
        hiLoad = hiLoad > ctx.orgHi ? ctx.orgHi : hiLoad;
        wiLoad = paddingWi - padLeftL1 - padRightL1;
    }

    uint8_t padList[PAD_SIZE] = {MAX_PAD_R, MAX_PAD_R, MAX_PAD_R, MAX_PAD_R};
    if (unlikely(allPad)) {
        Load3DSetFMatrixCal(MIN_HI_WI, MIN_HI_WI, padList);
    } else {
        padList[PAD_IDX_L] = padLeftL1;
        padList[PAD_IDX_R] = padRightL1;
        padList[PAD_IDX_T] = padTopL1;
        Load3DSetFMatrixCal(hiLoad, wiLoad, padList);
    }
    Load3DSetPaddingCal(t->offsetx);

    if (allPad) {
        InitConstValueParams<typename Ctx::FmapT> params(
            1, static_cast<uint16_t>(t->aL1SpaceSize / C0_SIZE), 0, 0);
        InitConstValue<typename Ctx::FmapT>(ctx.al1, params);
        ctx.queueAL1.EnQue(ctx.al1);
        return;
    }

    int64_t realHiTopGm = hiTop < 0 ? 0 : hiTop;
    int64_t realWiTopGm = wiLeft < 0 ? 0 : wiLeft;
    int64_t aL1GmOffset = realHiTopGm * ctx.orgWi + realWiTopGm;

    Dn2NzParams params;
    if (likely(wiLoad == ctx.orgWi)) {
        params.dnNum = 1;
        params.nValue = hiLoad * wiLoad;
        params.srcDnMatrixStride = 0;
        params.dstNzMatrixStride = 0;
    } else {
        params.dnNum = hiLoad;
        params.nValue = wiLoad;
        params.srcDnMatrixStride = ctx.orgWi;
        params.dstNzMatrixStride = wiLoad * Ctx::k0;
    }
    params.dValue = t->cinAInCore;
    params.srcDValue = t->orgHixWi;
    params.dstNzC0Stride = hiLoad * wiLoad;
    params.dstNzNStride = 1;
    DataCopy<typename Ctx::FmapT>(ctx.al1, ctx.agm[aL1GmOffset], params);
    ctx.queueAL1.EnQue(ctx.al1);
}

template <class Ctx>
__aicore__ inline void LoadBL1(Ctx &ctx)
{
    ctx.bl1 = ctx.queueBL1.template AllocTensor<typename Ctx::WeightT>();

    const auto *t = ctx.convTiling;
    bool oneByOne = (t->kernelHxkernelW == 1);
    if (oneByOne) {
        Nd2NzParams p;
        p.ndNum = 1;
        p.nValue = ctx.currentNBL1;
        p.dValue = t->kBL1;
        p.srcNdMatrixStride = 0;
        p.srcDValue = ctx.orgCi;
        p.dstNzC0Stride = t->nBL1;
        p.dstNzNStride = 1;
        p.dstNzMatrixStride = 0;
        DataCopy<typename Ctx::WeightT>(ctx.bl1, ctx.bgm[0], p);
    } else {
        Dn2NzParams p;
        p.dnNum = ctx.currentNBL1;
        p.nValue = t->kernelHxkernelW;
        p.dValue = t->cinBInCore;
        p.srcDnMatrixStride = t->coutOffsetBlock;
        p.srcDValue = t->kernelHxkernelW;
        p.dstNzC0Stride = t->kernelHxkernelW * t->nBL1;
        p.dstNzNStride = t->nBL1;
        p.dstNzMatrixStride = Ctx::k0;
        DataCopy<typename Ctx::WeightT>(ctx.bl1, ctx.bgm[0], p);
    }
    ctx.queueBL1.EnQue(ctx.bl1);
}



### 3.6 im2col：LoadAL0 / LoadBL0

**LoadAL0**：将 L1 中的特征图 patch 通过 `Load3D` 指令在线展开为 im2col 矩阵，写入 L0A。

`Load3D` 是 NPU 的专用硬件指令，能在数据搬运的同时完成 im2col 展开，无需额外的软件循环。第一次调用（`isFirst == true`）需要设置完整的卷积参数（步长、卷积核尺寸、膨胀系数、通道数），后续迭代只需更新 K 偏移量。

#### Load3D详解

**功能说明**：Load3D用于完成image to column操作，将多维feature map转为二维矩阵。支持如下数据通路：A1→A2; B1→B2。

**Load3Dv2接口**：

```cpp

template <typename T, 
          const IsResetLoad3dConfig &defaultConfig = IS_RESER_LOAD3D_DEFAULT_CONFIG,
          typename U = PrimT<T>,
          typename Std::enable_if<Std::is_same<PrimT<T>, U>::value, bool>::type = true>
__aicore__ inline void LoadData(const LocalTensor<T>& dst, 
                                const LocalTensor<T>& src, 
                                const LoadData3DParamsV2<U>& loadDataParams)

```
**参数说明**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>参数名称</strong></td>
  <td align="center"><strong>输入/输出</strong></td>
  <td align="center"><strong>含义</strong></td>
</tr>
<tr>
  <td align="left">dst</td>
  <td align="left">输出</td>
  <td align="left">目的操作数(LocalTensor)</td>
</tr>
<tr>
  <td align="left">src</td>
  <td align="left">输入</td>
  <td align="left">源操作数(LocalTensor/GlobalTensor)，类型需与dst保持一致</td>
</tr>
<tr>
  <td align="left">loadDataParams</td>
  <td align="left">输入</td>
  <td align="left">LoadData参数结构体(LoadData3DParamsV2)</td>
</tr>
</table>

**LoadData3DParamsV2 结构体参数**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>参数名称</strong></td>
  <td align="center"><strong>含义</strong></td>
  <td align="center" style="width: 180px"><strong>取值范围</strong></td>
  <td align="center" style="width: 80px"><strong>默认值</strong></td>
</tr>
<tr>
  <td align="left">padList</td>
  <td align="left">padding列表</td>
  <td align="left">[0,255]</td>
  <td align="left">{0,0,0,0}</td>
</tr>
<tr>
  <td align="left">l1H</td>
  <td align="left">源操作数height</td>
  <td align="left">[1, 32767]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">l1W</td>
  <td align="left">源操作数width</td>
  <td align="left">[1, 32767]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">channelSize</td>
  <td align="left">源操作数的通道数</td>
  <td align="left">[1, 63]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">kExtension</td>
  <td align="left">目的操作数width维度传输长度</td>
  <td align="left">[1, 65535]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">mExtension</td>
  <td align="left">目的操作数height维度传输长度</td>
  <td align="left">[1, 65535]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">kStartPt</td>
  <td align="left">目的操作数width维度起点</td>
  <td align="left">[0, 65535]</td>
  <td align="left">0</td>
</tr>
<tr>
  <td align="left">mStartPt</td>
  <td align="left">目的操作数height维度起点</td>
  <td align="left">[0, 65535]</td>
  <td align="left">0</td>
</tr>
<tr>
  <td align="left">strideW</td>
  <td align="left">卷积核w维度滑动步长</td>
  <td align="left">[1, 63]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">strideH</td>
  <td align="left">卷积核h维度滑动步长</td>
  <td align="left">[1, 63]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">filterW</td>
  <td align="left">卷积核width</td>
  <td align="left">[1, 255]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">filterH</td>
  <td align="left">卷积核height</td>
  <td align="left">[1, 255]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">dilationFilterW</td>
  <td align="left">卷积核width膨胀系数</td>
  <td align="left">[1, 255]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">dilationFilterH</td>
  <td align="left">卷积核height膨胀系数</td>
  <td align="left">[1, 255]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">enTranspose</td>
  <td align="left">是否启用转置功能(仅A2位置，half类型有效)</td>
  <td align="left">true/false</td>
  <td align="left">false</td>
</tr>
<tr>
  <td align="left">enSmallK</td>
  <td align="left">是否使能small k特性(已不再支持)</td>
  <td align="left">true/false</td>
  <td align="left">false</td>
</tr>
<tr>
  <td align="left">padValue</td>
  <td align="left">Pad填充值数值</td>
  <td align="left">与src数据类型一致</td>
  <td align="left">0</td>
</tr>
<tr>
  <td align="left">filterSizeW</td>
  <td align="left">是否在filterW基础上增加256个元素</td>
  <td align="left">true/false</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">filterSizeH</td>
  <td align="left">是否在filterH基础上增加256个元素</td>
  <td align="left">true/false</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">fMatrixCtrl</td>
  <td align="left">从左矩阵还是右矩阵获取FeatureMap属性描述(当前只支持false)</td>
  <td align="left">true/false</td>
  <td align="left">false</td>
</tr>
</table>

**LoadBL0**：将 L1 中的权重通过 `Load2D` 指令搬运到 L0B，转换为 NZ 布局。

第一次调用需要设置 N/K 步长和源矩阵步长，后续迭代只需更新 K 起始位置。

#### Load2D详解

**功能定位**：Load2D用于将权重数据从L1或GM搬运至L0B，支持矩阵转置和分块搬运，适配Cube单元的输入格式要求。

**Load2D接口**：

```cpp

template <typename T>
__aicore__ inline void LoadData(const LocalTensor<T>& dst, const LocalTensor<T>& src, const LoadData2DParams& loadDataParams)

```

**通用参数说明**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>参数名称</strong></td>
  <td align="center" style="width: 100px"><strong>输入/输出</strong></td>
  <td align="center"><strong>含义</strong></td>
</tr>
<tr>
  <td align="left">dst</td>
  <td align="left">输出</td>
  <td align="left">目的操作数，类型为LocalTensor。数据连续排列顺序由目的操作数所在TPosition决定：<br>- A2：ZZ格式，分形大小为16 * (32B / sizeof(T))<br>- B2：ZN格式，分形大小为(32B / sizeof(T)) * 16<br>- A1/B1：无格式要求，一般为NZ格式，分形大小为16 * (32B / sizeof(T))</td>
</tr>
<tr>
  <td align="left">src</td>
  <td align="left">输入</td>
  <td align="left">源操作数，类型为LocalTensor或GlobalTensor。数据类型需与dst保持一致。</td>
</tr>
<tr>
  <td align="left">loadDataParams</td>
  <td align="left">输入</td>
  <td align="left">LoadData参数结构体，类型为LoadData2DParams。</td>
</tr>
</table>

**LoadData2DParams结构体内参数说明**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>参数名称</strong></td>
  <td align="center"><strong>含义</strong></td>
  <td align="center" style="width: 180px"><strong>取值范围</strong></td>
  <td align="center" style="width: 80px"><strong>默认值</strong></td>
</tr>
<tr>
  <td align="left">startIndex</td>
  <td align="left">分形矩阵ID，说明搬运起始位置为源操作数中第几个分形（0为第1个分形矩阵）。单位：512B。</td>
  <td align="left">[0, 65535]</td>
  <td align="left">0</td>
</tr>
<tr>
  <td align="left">repeatTimes</td>
  <td align="left">迭代次数，每个迭代可以处理512B数据。</td>
  <td align="left">[1, 255]</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">srcStride</td>
  <td align="left">相邻迭代间，源操作数前一个分形与后一个分形起始地址的间隔。单位：512B。</td>
  <td align="left">[0, 65535]</td>
  <td align="left">0</td>
</tr>
<tr>
  <td align="left">sid</td>
  <td align="left">预留参数，配置为0即可。</td>
  <td align="left">-</td>
  <td align="left">-</td>
</tr>
<tr>
  <td align="left">dstGap</td>
  <td align="left">相邻迭代间，目的操作数前一个分形结束地址与后一个分形起始地址的间隔。单位：512B。<br><strong>注</strong>：Atlas 训练系列产品此参数不使能。</td>
  <td align="left">[0, 65535]</td>
  <td align="left">0</td>
</tr>
<tr>
  <td align="left">ifTranspose</td>
  <td align="left">是否启用转置功能，对每个分形矩阵进行转置。<br><strong>注意</strong>：只有A1->A2和B1->B2通路才能使能转置，使能转置时仅支持uint16_t/int16_t/half数据类型。</td>
  <td align="left">true/false</td>
  <td align="left">false</td>
</tr>
<tr>
  <td align="left">addrMode</td>
  <td align="left">预留参数，配置为0即可。</td>
  <td align="left">-</td>
  <td align="left">-</td>
</tr>
</table>

**约束说明**：

- 操作数地址对齐要求请参见通用地址对齐约束。
- repeatTimes最大值为255，当搬运数据量超过255块时需分批搬运。

**repeatTimes限制处理方案**：

当搬运数据量超过255块时，需要分批搬运：

```cpp

if (repeatTimes > LOAD2D_MAX_REPEAT_TIMES) {
    // 方案一：使用DataCopy替代
    DataCopyParams dataCopyParams(1, blockLen, 0, 0);
    DataCopy<WeightT>(bl1, bgm[offset], dataCopyParams);
    
    // 方案二：分批LoadData2D
    for (i = 0; i < repeatTimes / 255; i++) {
        LoadData<WeightT>(bl1[offset1], bgm[offset2], params255);
    }
    // 处理尾部
    if (tail > 0) {
        LoadData<WeightT>(bl1[offset1], bgm[offset2], paramsTail);
    }
}

```


In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
template <class Ctx>
__aicore__ inline void LoadAL0(Ctx &ctx, bool isFirst)
{
    if ASCEND_IS_AIV {
        return;
    }
    const auto *t = ctx.convTiling;
    uint64_t currentKL0 = (ctx.kIter == ctx.maxKL0Iter) ? ctx.kL0Tail : t->kL0;
    uint64_t posK = ctx.kAL0Iter * t->kL0;
    uint64_t channelSize = t->cinAInCore;
    uint64_t currentML0 = ctx.currentHoL0 * ctx.currentWoL0;
    uint16_t mExtension = static_cast<uint16_t>(currentML0);
    uint16_t mStartPt = 0;

    Load3DBitModeParam param;
    if (isFirst) {
        uint64_t xmtmp = ((mExtension & MASK_16) << MSTEP_OFFSET) |
                         ((uint64_t)(mStartPt & MASK_16) << POSM_OFFSET);
        uint64_t xt = ((uint64_t(t->strideW) & MASK_6) << 0) |
                      ((uint64_t(t->strideH) & MASK_6) << STRIDEH_OFFSET) |
                      ((uint64_t(ctx.kernelW) & MASK_8) << KERNELW_OFFSET) |
                      ((uint64_t(ctx.kernelW) & NINTH_BIT_MASK) << KERNELW_HIGHEST_BIT_OFFSET) |
                      ((uint64_t(ctx.kernelH) & MASK_8) << KERNELH_OFFSET) |
                      ((uint64_t(ctx.kernelH) & NINTH_BIT_MASK) << KERNELH_HIGHEST_BIT_OFFSET) |
                      ((uint64_t(t->dilationW) & MASK_8) << DILATIONW_OFFSET) |
                      ((uint64_t(t->dilationH) & MASK_8) << DILATIONH_OFFSET) |
                      ((uint64_t(channelSize) & MASK_16) << CIN_OFFSET);
        param.SetConfig1(xt);
        uint64_t xm = ((currentKL0 & MASK_16) << 0) | ((posK & MASK_16) << POSK_OFFSET) | xmtmp;
        param.SetConfig0(xm);

        uint16_t dstStride =
            (ctx.currentWoL0 == t->woL0 && ctx.currentHoL0 == t->hoL0)
                ? static_cast<uint16_t>(t->fmapKStride)
                : static_cast<uint16_t>(ctx.currentML0Align / BLOCK_L0_M);
        LoadDataRepeatParam repeatParams = {0, 1, 0, dstStride};
        SetLoadDataRepeat(repeatParams);
    } else {
        uint64_t xmtmp = ((mExtension & MASK_16) << MSTEP_OFFSET) |
                         ((uint64_t)(mStartPt & MASK_16) << POSM_OFFSET);
        uint64_t xm = ((currentKL0 & MASK_16) << 0) | ((posK & MASK_16) << POSK_OFFSET) | xmtmp;
        param.SetConfig0(xm);
    }
    LoadData<TPosition::A2, TPosition::A1, typename Ctx::FmapT>(ctx.al0, ctx.al1, param);
}



### 3.7 矩阵乘累加：Mad 与 CopyOut

**Mad**：调用 `Mmad` 指令执行矩阵乘累加，将 L0A（特征图 im2col）× L0B（权重 NZ）的结果累加到 L0C（fp32 累加器）。

关键参数：
- `cmatrixInitVal = (kIter == 0)`：第一次 K 迭代时清零 L0C，后续迭代累加。
- `unitFlag`：最后一次 K 迭代时设置 `UNIT_FLAG_ENABLE_WITH_FLIP`，触发 L0C 翻转（通知 Fixpipe 可以读取结果）。

**CopyOut**：调用 `Fixpipe` 指令将 L0C 中的 fp32 结果转换为 fp16，并以 NCHW 布局写回 GM。

`FixpipeParamsC310` 中的 `quantPre = F322F16` 指定了量化模式，`deqScalar = DEQ_SCALAR_ONE`（即浮点 1.0）表示不缩放。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
template <class Ctx>
__aicore__ inline void LoadBL0(Ctx &ctx, bool isFirst)
{
    const auto *t = ctx.convTiling;
    uint64_t kStep = (ctx.kIter != ctx.maxKL0Iter) ? t->kStep : (ctx.kL0Tail / Ctx::k0);
    uint64_t ratioOfNToN0 = ctx.currentNL0Align / BLOCK_L0_N;

    Load2DBitModeParam param;
    if (unlikely(isFirst)) {
        param.SetMStartPosition(0);
        param.SetKStartPosition(static_cast<uint32_t>(ctx.kBL0Iter * t->kStep));
        param.SetMStep(static_cast<uint16_t>(ratioOfNToN0));
        param.SetKStep(static_cast<uint16_t>(kStep));
        param.SetSrcStride(static_cast<int32_t>(t->nL1DivBlockSize));
        param.SetDstStride(static_cast<uint16_t>(ratioOfNToN0));
        param.SetIfTranspose(false);
    } else {
        param.SetKStartPosition(static_cast<uint32_t>(ctx.kBL0Iter * t->kStep));
        param.SetKStep(static_cast<uint16_t>(kStep));
    }
    LoadData<TPosition::B2, TPosition::B1, typename Ctx::WeightT>(ctx.bl0, ctx.bl1, param);
}



### 3.8 迭代控制：IterateOne 与 EndConv

**IterateOne**：执行一次完整的卷积 tile 计算，包含以下流水步骤：

1. 调用 `LoadAL1` / `LoadBL1` 将数据从 GM 搬运到 L1。
2. 进入 K 维度循环：每次迭代调用 `LoadAL0`（im2col）、`LoadBL0`（权重 NZ）、`Mad`（矩阵乘累加）。
3. 使用 `WaitFlag` / `SetFlag` 在 MTE1（数据搬运）和 M（矩阵计算）单元之间进行同步。
4. 调用 `CopyOut` 将 L0C 结果写回 GM。

**EndConv**：释放 L1 队列中的所有事件资源，完成一次卷积的收尾工作。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
template <class Ctx>
__aicore__ inline void Mad(Ctx &ctx)
{
    MmadParams p;
    p.m = ctx.currentML0Align;
    p.n = ctx.currentNL0Align;
    p.k = (ctx.kIter == ctx.maxKL0Iter) ? ctx.kL0Tail : ctx.convTiling->kL0;
    p.cmatrixInitVal = (ctx.kIter == 0);
    p.cmatrixSource = false;
    p.unitFlag = (ctx.kIter == ctx.maxKL0Iter) ? UNIT_FLAG_ENABLE_WITH_FLIP : UNIT_FLAG_ENABLE_ONLY;
    Mmad<typename Ctx::L0cT, typename Ctx::FmapT, typename Ctx::WeightT>(
        ctx.cl0, ctx.al0, ctx.bl0, p);
}

template <class Ctx>
__aicore__ inline void CopyOut(Ctx &ctx, const GlobalTensor<typename Ctx::OutputT> &output)
{
    uint64_t valueHoWo = ctx.orgHo * ctx.orgWo;
    uint64_t offset = 0;

    FixpipeParamsC310<CO2Layout::COLUMN_MAJOR> p;
    p.mSize = ctx.currentHoL0 * ctx.currentWoL0;
    p.params.srcNzMatrixStride = 0;
    p.params.dnNum = 1;
    p.params.dstDnMatrixStride = 0;
    p.nSize = ctx.currentNL0Align;
    p.srcStride = AlignB(p.mSize, BLOCK_L0_M);
    p.dstStride = valueHoWo;
    p.params.srcNzC0Stride = 1;
    p.quantPre = QuantMode_t::F322F16;
    p.deqScalar = DEQ_SCALAR_ONE;
    p.unitFlag = UNIT_FLAG_ENABLE_WITH_FLIP;

    Fixpipe<typename Ctx::OutputT, typename Ctx::L0cT, CFG_COLUMN_MAJOR>(
        output[offset], ctx.cl0, p);
}



### 3.9 Conv2dBase 类与核函数入口

**关闭 namespace conv**，然后定义三个 `constexpr` 变量指定特征图、权重和输出的数据布局（均为 NCHW）。

**Conv2dBase**：模板类，封装了 `RunConv2dKernel` 方法。该方法：
1. 设置 GM 指针（特征图、权重、输出）。
2. 调用 `InitContext` 初始化上下文。
3. 调用 `IterateOne` 执行卷积计算。
4. 调用 `EndConv` 释放资源。

**conv2d_v2_kernel**：全局核函数入口，使用 `GET_TILING_DATA` 宏从 GM 读取 Tiling 参数，实例化类型标签，创建 `Conv2dBase` 对象并调用 `RunConv2dKernel`。

`#endif` 结束 `__CCE_AICORE__` 设备侧编译块。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
template <class Ctx>
__aicore__ inline void IterateOne(Ctx &ctx, const GlobalTensor<typename Ctx::OutputT> &output)
{
    ctx.cl0 = ctx.wholeCl0Tensor;
    ctx.kIter = 0;

    LoadAL1(ctx);
    LoadBL1(ctx);
    ctx.al1 = ctx.queueAL1.template DeQue<typename Ctx::FmapT>();
    ctx.bl1 = ctx.queueBL1.template DeQue<typename Ctx::WeightT>();

    event_t ev = static_cast<event_t>(L0_SYBC_DB_CLOSE);
    bool isFirst = true;
    while (ctx.kIter < ctx.ddr2l0LoopK) {
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(ev);
        ctx.al0 = ctx.wholeAl0Tensor;
        ctx.kAL0Iter = ctx.kIter;
        LoadAL0(ctx, isFirst);
        ctx.bl0 = ctx.wholeBl0Tensor;
        ctx.kBL0Iter = ctx.kIter;
        LoadBL0(ctx, isFirst);
        AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(ev);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(ev);
        Mad(ctx);
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(ev);
        ctx.kIter++;
        isFirst = false;
    }

    ctx.queueAL1.FreeTensor(ctx.al1);
    ctx.queueBL1.FreeTensor(ctx.bl1);

    if ASCEND_IS_AIC_CONV {
        CopyOut(ctx, output);
    }
}

template <class Ctx>
__aicore__ inline void EndConv(Ctx &ctx)
{
    if ASCEND_IS_AIC_CONV {
        ctx.queueAL1.FreeAllEvent();
        ctx.queueBL1.FreeAllEvent();
    }
}



---

## 4. Host侧调用代码

Host侧代码负责：填充 Tiling 结构体、验证片上空间、分配设备内存、启动核函数、回收结果。

### 4.1 卷积形状常量

定义本 Demo 使用的卷积形状参数。所有后续的 Tiling 计算和内存分配都基于这些常量。

`Fp16ToFloat` 是一个纯软件的 fp16 解码函数，用于在主机侧将输出结果转换为 float 以便打印。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
}  // namespace conv

using namespace conv;

constexpr ConvFormat fmapFormat = ConvFormat::NCHW;
constexpr ConvFormat filterFormat = ConvFormat::NCHW;
constexpr ConvFormat outputFormat = ConvFormat::NCHW;

template <class FMAP_TYPE, class WEIGHT_TYPE, class OUTPUT_TYPE>
class Conv2dBase {
public:
    using FMAP_T = typename FMAP_TYPE::T;
    using WEIGHT_T = typename WEIGHT_TYPE::T;
    using OUTPUT_T = typename OUTPUT_TYPE::T;

    __aicore__ inline void RunConv2dKernel(GM_ADDR x, GM_ADDR filter, GM_ADDR y,
                                           const Conv2DTilingData &tilingData)
    {
        ctx.hiStartPos = -static_cast<int64_t>(tilingData.padTop);
        ctx.wiStartPos = -static_cast<int64_t>(tilingData.padLeft);
        ctx.hiStartPos = ctx.hiStartPos > 0 ? 0 : ctx.hiStartPos;

        fmapGm.SetGlobalBuffer(reinterpret_cast<__gm__ FMAP_T *>(x));
        filterGm.SetGlobalBuffer(reinterpret_cast<__gm__ WEIGHT_T *>(filter));
        outputGm.SetGlobalBuffer(reinterpret_cast<__gm__ OUTPUT_T *>(y));

        InitContext(ctx, &tilingData);
        ctx.hiStartPos = -static_cast<int64_t>(tilingData.padTop);
        ctx.wiStartPos = -static_cast<int64_t>(tilingData.padLeft);

        ctx.agm = fmapGm;
        ctx.bgm = filterGm;

        if ASCEND_IS_AIC_CONV {
            IterateOne(ctx, outputGm);
        }
        EndConv(ctx);
    }

private:
    conv::Conv2dContext<FMAP_TYPE, WEIGHT_TYPE, OUTPUT_TYPE> ctx;
    GlobalTensor<FMAP_T> fmapGm;
    GlobalTensor<WEIGHT_T> filterGm;
    GlobalTensor<OUTPUT_T> outputGm;
};
#endif



### 4.2 卷积形状常量与辅助函数

定义本 Demo 使用的卷积形状参数（BATCH、CI、HI、WI、CO、KH、KW 等）。所有后续的 Tiling 计算和内存分配都基于这些常量。

`Fp16ToFloat` 是一个纯软件的 fp16 解码函数，用于在主机侧将输出结果转换为 float 以便打印。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
constexpr uint32_t BATCH = 1;
constexpr uint32_t CI = 16;   // input channels
constexpr uint32_t HI = 8;    // input height
constexpr uint32_t WI = 8;    // input width
constexpr uint32_t CO = 16;   // output channels (must be multiple of 16 for NZ layout)
constexpr uint32_t KH = 3;    // kernel height
constexpr uint32_t KW = 3;    // kernel width
constexpr uint32_t STRIDE_H = 1;
constexpr uint32_t STRIDE_W = 1;
constexpr uint32_t DILATION_H = 1;
constexpr uint32_t DILATION_W = 1;
constexpr uint32_t PAD_TOP = 0;
constexpr uint32_t PAD_BOTTOM = 0;
constexpr uint32_t PAD_LEFT = 0;
constexpr uint32_t PAD_RIGHT = 0;
// Standard conv output size formula
constexpr uint32_t HO = (HI + PAD_TOP + PAD_BOTTOM - DILATION_H * (KH - 1) - 1) / STRIDE_H + 1;
constexpr uint32_t WO = (WI + PAD_LEFT + PAD_RIGHT - DILATION_W * (KW - 1) - 1) / STRIDE_W + 1;

__global__ __aicore__ void conv2d_v2_kernel(GM_ADDR x, GM_ADDR filter, GM_ADDR bias,
    GM_ADDR offset_w, GM_ADDR y, __kfc_workspace__ GM_ADDR workspace, GM_ADDR tilingGm)
{
#ifdef __CCE_AICORE__
    GET_TILING_DATA(tilingData, tilingGm);

    using fmapType = ConvType<TPosition::GM, fmapFormat, half>;
    using weightType = ConvType<TPosition::GM, filterFormat, half>;
    using outputType = ConvType<TPosition::GM, outputFormat, half>;

    Conv2dBase<fmapType, weightType, outputType> baseConv2d;
    baseConv2d.RunConv2dKernel(x, filter, y, tilingData);
#endif
}

constexpr uint8_t BLOCK_SIZE = 16;

float Fp16ToFloat(uint16_t h)
{
    uint32_t sign = (h >> 15) & 0x1;
    uint32_t exp = (h >> 10) & 0x1F;
    uint32_t frac = h & 0x3FF;
    float result;
    if (exp == 0) {
        if (frac == 0) {
            result = 0.0f;
        } else {
            result = std::ldexp(static_cast<float>(frac), -24);
        }
    } else if (exp == 31) {
        result = (frac == 0) ? std::numeric_limits<float>::infinity() : std::numeric_limits<float>::quiet_NaN();
    } else {
        result = std::ldexp(1.0f + static_cast<float>(frac) / 1024.0f, static_cast<int>(exp) - 15);
    }
    return sign ? -result : result;
}



### 4.3 SetTilingData

`SetTilingData` 根据卷积形状常量填充 `Conv2DTilingData` 结构体。

关键字段说明：
- **kAL1 / kBL1**：L1 侧 K 维度大小 = KH × KW × CI（一次性加载全部 K）。
- **kL0**：L0 侧 K 维度大小，本 Demo 中等于 kAL1（单次 Mmad 完成全部 K 累加）。
- **mStep**：M 维度对齐到 16 的倍数（Mmad 要求 M 为 BLOCK_L0_M 的整数倍）。
- **cinAInCore / cinBInCore**：每核处理的输入通道数（= kAL1 / kernelHW）。
- **nL1DivBlockSize**：L1 中 N 方向的块数（= nBL1 / BLOCK_SIZE）。

代码中标注了 `[QUIZ]` 的位置是本章课后练习的填空点。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
void SetTilingData(Conv2DTilingData* tiling)
{
    tiling->padTop = PAD_TOP;
    tiling->padLeft = PAD_LEFT;

    // ── ApiTiling: memory-hierarchy tiling parameters ──────────────────────
    // These tell the kernel how large each on-chip buffer tile is.
    // For this full-load demo the entire output tile fits in one L0 block,
    // so hoL0/woL0 >= singleCoreHo/Wo and kL0 == kAL1 == kBL1 == total K.
    tiling->orgHi = HI;
    tiling->orgWi = WI;
    tiling->orgHo = HO;
    tiling->orgWo = WO;
    tiling->singleCoreBatch = BATCH;
    tiling->singleCoreHo = HO;
    tiling->singleCoreWo = WO;
    tiling->orgCi = CI;
    tiling->orgCo = CO;
    tiling->singleCoreCi = CI;
    tiling->singleCoreCo = CO;
    // [QUIZ 1/4] kAL1: total K dimension loaded into L1 for the fmap side.
    //   K = number of multiply-accumulate steps per output pixel
    //     = kernel_h × kernel_w × Cin
    //   Hint: use KH, KW, CI
    /* BEGIN_TODO */
    tiling->kAL1 = KH * KW * CI;
    /* END_TODO */
    // [QUIZ 2/4] kBL1: same K dimension for the weight side (must equal kAL1).
    /* BEGIN_TODO */
    tiling->kBL1 = KH * KW * CI;
    /* END_TODO */
    tiling->nBL1 = 16;
    tiling->hoL0 = 16;
    tiling->woL0 = WO;
    // [QUIZ 3/4] kL0: K dimension actually loaded into L0A/L0B per Mmad call.
    //   In this full-load demo kL0 == kAL1 (one shot, no K-loop).
    /* BEGIN_TODO */
    tiling->kL0 = KH * KW * CI;
    /* END_TODO */
    tiling->nL0 = 16;
    tiling->orgHixWi = HI * WI;
    tiling->kernelHxkernelW = KH * KW;
    tiling->aL1SpaceSize = 8192;

    // Derived fields — computed from the shape constants above
    uint32_t kernelHW = KH * KW;
    tiling->cinAInCore = tiling->kAL1 / kernelHW;
    tiling->cinBInCore = tiling->kBL1 / kernelHW;
    // [QUIZ 4/4] mStep: M dimension aligned up to the next multiple of 16.
    //   M = total output pixels per core = HO × WO
    //   Mmad requires M to be a multiple of BLOCK_L0_M (16).
    //   Hint: AlignUp(HO * WO, 16)
    /* BEGIN_TODO */
    tiling->mStep = ((HO * WO + 15) / 16) * 16;
    /* END_TODO */
    tiling->kStep = 1;
    tiling->fmapKStride = 1;
    tiling->coutOffsetBlock = CI * kernelHW;
    tiling->nL1DivBlockSize = tiling->nBL1 / BLOCK_SIZE;
    tiling->kernelH = KH;
    tiling->kernelW = KW;
    tiling->strideH = STRIDE_H;
    tiling->strideW = STRIDE_W;
    tiling->dilationH = DILATION_H;
    tiling->dilationW = DILATION_W;
    tiling->offsetx = 0;
}

// ============================================================================
// Space validation
// ============================================================================
//
// Before launching the kernel, verify that the tiling parameters actually fit
// in the NPU's on-chip SRAM.  If they don't, the kernel will silently corrupt
// memory or produce wrong results — there is no hardware bounds check.
//
// On-chip buffer sizes (Ascend 950PR/Ascend 950DT):
//   L1  : 512 MB  per core  (fmap + weight staging area)
//   L0A : 64 KB per core  (matrix A — im2col output)
//   L0B : 64 KB per core  (matrix B — weight NZ)
//   L0C : 256 KB per core (accumulator, fp32)
//
// Layout sizes:
//   AL1 : hiLoad * wiLoad * CI * sizeof(fp16)
//         where hiLoad = (HO-1)*strideH + dilatedKH
//               wiLoad = (WO-1)*strideW + dilatedKW
//   BL1 : nBL1 * kBL1 * sizeof(fp16)
//   AL0 : AlignUp(HO*WO, 16) * kL0 * sizeof(fp16)   [NZ: M-blocks × K]
//   BL0 : AlignUp(CO, 16)    * kL0 * sizeof(fp16)   [NZ: N-blocks × K]
//   CL0 : AlignUp(HO*WO, 16) * AlignUp(CO, 16) * sizeof(fp32)
//


### 4.4 CheckTilingSpace

在启动核函数之前，必须验证 Tiling 参数是否真正适合 NPU 的片上 SRAM。若超出限制，核函数会静默地越界访问片上内存，产生错误结果或硬件异常——没有任何硬件边界检查。

各缓冲区的大小限制（Ascend 950PR/Ascend 950DT）：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>缓冲区</strong></td>
  <td align="center"><strong>大小</strong></td>
</tr>
<tr>
  <td align="left"> L1 </td>
  <td align="left">512 MB / 核</td>
</tr>
<tr>
  <td align="left">L0A  </td>
  <td align="left">  64 KB / 核 </td>
</tr>
<tr>
  <td align="left">L0B  </td>
  <td align="left">64 KB / 核 </td>
</tr>
<tr>
  <td align="left"> L0C</td>
  <td align="left"> 256 KB / 核</td>
</tr>
</table>

代码中标注了 `[QUIZ 1/5]` 到 `[QUIZ 5/5]` 的位置是本章课后练习的填空点。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
static bool CheckTilingSpace(const Conv2DTilingData* a)
{
    const uint32_t fp16 = sizeof(uint16_t);
    const uint32_t fp32 = sizeof(float);
    const uint32_t align16 = 16;

    // L1 — fmap patch that load3d reads from
    // [QUIZ 1/5] Effective kernel size after dilation.
    //   dilatedKH = 1 + (KH - 1) * DILATION_H
    //   (a 3×3 kernel with dilation=2 spans 5 rows in the input)
    /* BEGIN_TODO */
    uint32_t dilatedKH = 1 + (KH - 1) * DILATION_H;
    uint32_t dilatedKW = 1 + (KW - 1) * DILATION_W;
    /* END_TODO */
    // [QUIZ 2/5] Input patch height/width that L1 must hold for one output tile.
    //   The last output row at index (HO-1) reads input row (HO-1)*strideH + (dilatedKH-1),
    //   so the patch spans: hiLoad = (HO-1)*STRIDE_H + dilatedKH rows.
    /* BEGIN_TODO */
    uint32_t hiLoad = (HO - 1) * STRIDE_H + dilatedKH;
    uint32_t wiLoad = (WO - 1) * STRIDE_W + dilatedKW;
    /* END_TODO */
    // [QUIZ 3/5] AL1 bytes: the fmap patch is hiLoad × wiLoad × CI fp16 elements.
    /* BEGIN_TODO */
    uint64_t al1Bytes = (uint64_t)hiLoad * wiLoad * CI * fp16;
    /* END_TODO */

    // L1 — weight block
    uint64_t bl1Bytes = (uint64_t)a->nBL1 * a->kBL1 * fp16;

    // L1 total
    const uint64_t L1_SIZE = 1024 * 1024ULL;
    uint64_t l1Total = al1Bytes + bl1Bytes;

    // L0A — im2col result: M-blocks × K elements, fp16
    // [QUIZ 4/5] AL0 bytes.
    //   After im2col, L0A holds mAlign × kL0 fp16 elements in NZ layout.
    //   mAlign = AlignUp(HO*WO, 16)  — M must be a multiple of 16 for Mmad.
    /* BEGIN_TODO */
    uint32_t mAlign = ((HO * WO + align16 - 1) / align16) * align16;
    uint64_t al0Bytes = (uint64_t)mAlign * a->kL0 * fp16;
    /* END_TODO */

    // L0B — weight NZ: N-blocks × K elements, fp16
    uint32_t nAlign = ((CO + align16 - 1) / align16) * align16;
    uint64_t bl0Bytes = (uint64_t)nAlign * a->kL0 * fp16;

    // L0C — accumulator: M-blocks × N-blocks, fp32
    // [QUIZ 5/5] CL0 bytes.
    //   L0C stores the Mmad result as fp32 (even though inputs are fp16).
    //   Size = mAlign × nAlign × sizeof(float).
    //   Note: fp32 = 4 bytes, NOT fp16 = 2 bytes — this is the most common mistake.
    /* BEGIN_TODO */
    uint64_t cl0Bytes = (uint64_t)mAlign * nAlign * fp32;
    /* END_TODO */

    const uint64_t L0A_LIMIT = 64 * 1024ULL;
    const uint64_t L0B_LIMIT = 64 * 1024ULL;
    const uint64_t L0C_LIMIT = 256 * 1024ULL;

    bool ok = true;
    if (l1Total > L1_SIZE) {
        std::cerr << "[TILING ERROR] L1 overflow: need " << l1Total
                  << " bytes (AL1=" << al1Bytes << " + BL1=" << bl1Bytes
                  << "), limit=" << L1_SIZE << "\n";
        ok = false;
    }
    if (al0Bytes > L0A_LIMIT) {
        std::cerr << "[TILING ERROR] L0A overflow: need " << al0Bytes
                  << " bytes (M=" << mAlign << " K=" << a->kL0
                  << "), limit=" << L0A_LIMIT << "\n";
        ok = false;
    }
    if (bl0Bytes > L0B_LIMIT) {
        std::cerr << "[TILING ERROR] L0B overflow: need " << bl0Bytes
                  << " bytes (N=" << nAlign << " K=" << a->kL0
                  << "), limit=" << L0B_LIMIT << "\n";
        ok = false;
    }
    if (cl0Bytes > L0C_LIMIT) {
        std::cerr << "[TILING ERROR] L0C overflow: need " << cl0Bytes
                  << " bytes (M=" << mAlign << " N=" << nAlign
                  << "), limit=" << L0C_LIMIT << "\n";
        ok = false;
    }
    if (ok) {
        std::cout << "[TILING OK] L1=" << l1Total << "/" << L1_SIZE
                  << "  L0A=" << al0Bytes << "/" << L0A_LIMIT
                  << "  L0B=" << bl0Bytes << "/" << L0B_LIMIT
                  << "  L0C=" << cl0Bytes << "/" << L0C_LIMIT << "\n";
    }
    return ok;
}



### 4.5 main 函数

`main` 函数完整演示了 AscendC 核函数的调用生命周期：

1. 调用 `SetTilingData` 填充 Tiling 结构体。
2. 调用 `CheckTilingSpace` 验证片上空间，若溢出则提前退出。
3. 初始化 ACL 运行环境（`aclInit` → `aclrtSetDevice` → `aclrtCreateStream`）。
4. 分配主机和设备内存，将输入数据（全 1.0 的 fp16）从主机复制到设备。
5. 通过 `<<<numBlocks, nullptr, stream>>>` 语法启动核函数，并用 `aclrtSynchronizeStream` 等待完成。
6. 将输出从设备复制回主机，并写入文本文件。
7. 释放所有内存和运行时资源。

In [ ]:
%%writefile -a Sources/01.03/minimal_demo.asc
int32_t main(int32_t argc, char* argv[])
{
    size_t inputSize = BATCH * CI * HI * WI * sizeof(uint16_t);
    size_t filterSize = CO * CI * KH * KW * sizeof(uint16_t);
    size_t outputSize = BATCH * CO * HO * WO * sizeof(uint16_t);
    size_t tilingSize = sizeof(Conv2DTilingData);
    size_t workspaceSize = 16 * 1024 * 1024;
    uint32_t numBlocks = 1;

    Conv2DTilingData tilingBuf;
    memset(&tilingBuf, 0, sizeof(Conv2DTilingData));
    SetTilingData(&tilingBuf);

    // Validate that the tiling fits in on-chip SRAM before touching the device.
    // An overflow here means the kernel would read/write out-of-bounds on-chip
    // memory, producing silent wrong results or a hardware exception.
    if (!CheckTilingSpace(&tilingBuf)) {
        std::cerr << "Tiling validation failed. Adjust shape constants or tiling params.\n";
        return 1;
    }

    aclInit(nullptr);
    int32_t deviceId = 0;
    aclrtSetDevice(deviceId);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    uint8_t* xHost;
    uint8_t* xDevice;
    aclrtMallocHost((void**)(&xHost), inputSize);
    aclrtMalloc((void**)&xDevice, inputSize, ACL_MEM_MALLOC_HUGE_FIRST);
    uint16_t* xHalf = reinterpret_cast<uint16_t*>(xHost);
    for (uint32_t i = 0; i < BATCH * CI * HI * WI; ++i) {
        xHalf[i] = 0x3C00; // fp16 1.0
    }
    aclrtMemcpy(xDevice, inputSize, xHost, inputSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t* filterHost;
    uint8_t* filterDevice;
    aclrtMallocHost((void**)(&filterHost), filterSize);
    aclrtMalloc((void**)&filterDevice, filterSize, ACL_MEM_MALLOC_HUGE_FIRST);
    uint16_t* filterHalf = reinterpret_cast<uint16_t*>(filterHost);
    for (uint32_t i = 0; i < CO * CI * KH * KW; ++i) {
        filterHalf[i] = 0x3C00; // fp16 1.0
    }
    aclrtMemcpy(filterDevice, filterSize, filterHost, filterSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t* yDevice;
    uint8_t* yHost;
    aclrtMallocHost((void**)(&yHost), outputSize);
    aclrtMalloc((void**)&yDevice, outputSize, ACL_MEM_MALLOC_HUGE_FIRST);

    uint8_t* workspaceDevice;
    aclrtMalloc((void**)&workspaceDevice, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST);

    uint8_t* tilingHost;
    uint8_t* tilingDevice;
    aclrtMallocHost((void**)(&tilingHost), tilingSize);
    aclrtMalloc((void**)&tilingDevice, tilingSize, ACL_MEM_MALLOC_HUGE_FIRST);
    memcpy(tilingHost, &tilingBuf, tilingSize);
    aclrtMemcpy(tilingDevice, tilingSize, tilingHost, tilingSize, ACL_MEMCPY_HOST_TO_DEVICE);

    conv2d_v2_kernel<<<numBlocks, nullptr, stream>>>(
        xDevice, filterDevice, nullptr, nullptr, yDevice, workspaceDevice, tilingDevice);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(yHost, outputSize, yDevice, outputSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::cout << "Conv2D V2 kernel execution completed." << std::endl;

    // Write input matrix to file
    {
        std::ofstream ofs("input_matrix.txt");
        uint16_t* xData = reinterpret_cast<uint16_t*>(xHost);
        ofs << "Input shape: [" << BATCH << ", " << CI << ", " << HI << ", " << WI << "]" << std::endl;
        for (uint32_t n = 0; n < BATCH; ++n) {
            for (uint32_t c = 0; c < CI; ++c) {
                ofs << "N=" << n << ", C=" << c << ":" << std::endl;
                for (uint32_t h = 0; h < HI; ++h) {
                    for (uint32_t w = 0; w < WI; ++w) {
                        uint32_t idx = ((n * CI + c) * HI + h) * WI + w;
                        ofs << Fp16ToFloat(xData[idx]) << " ";
                    }
                    ofs << std::endl;
                }
            }
        }
        ofs.close();
        std::cout << "Input matrix written to input_matrix.txt" << std::endl;
    }

    // Write output matrix to file
    {
        std::ofstream ofs("output_matrix.txt");
        uint16_t* yData = reinterpret_cast<uint16_t*>(yHost);
        ofs << "Output shape: [" << BATCH << ", " << CO << ", " << HO << ", " << WO << "]" << std::endl;
        for (uint32_t n = 0; n < BATCH; ++n) {
            for (uint32_t c = 0; c < CO; ++c) {
                ofs << "N=" << n << ", C=" << c << ":" << std::endl;
                for (uint32_t h = 0; h < HO; ++h) {
                    for (uint32_t w = 0; w < WO; ++w) {
                        uint32_t idx = ((n * CO + c) * HO + h) * WO + w;
                        ofs << Fp16ToFloat(yData[idx]) << " ";
                    }
                    ofs << std::endl;
                }
            }
        }
        ofs.close();
        std::cout << "Output matrix written to output_matrix.txt" << std::endl;
    }

    aclrtFree(xDevice);
    aclrtFreeHost(xHost);
    aclrtFree(filterDevice);
    aclrtFreeHost(filterHost);
    aclrtFree(yDevice);
    aclrtFreeHost(yHost);
    aclrtFree(workspaceDevice);
    aclrtFree(tilingDevice);
    aclrtFreeHost(tilingHost);

    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();

    return 0;
}


---

## 5. CMakeLists.txt

以下是本项目的 CMake 构建配置。与 Add 算子示例相比，Conv2D V2 Demo 需要额外链接多个 CANN 运行时库（`tiling_api`、`ascendc_runtime`、`ascendcl` 等），并通过 `target_include_directories` 指定 AscendC 内部头文件路径。


In [ ]:
%%writefile Sources/01.03/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

find_package(ASC REQUIRED)

project(conv2d_v2_demo_minimal LANGUAGES ASC CXX)

add_executable(minimal_demo
    minimal_demo.asc
)

target_include_directories(minimal_demo PRIVATE
    ${CMAKE_CURRENT_SOURCE_DIR}
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/asc/impl
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/asc/impl/basic_api
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/asc/impl/basic_api/reg_compute
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/asc/impl/simt_api
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/include
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/tikcpp/tikcfw
)

target_link_directories(minimal_demo PRIVATE
    ${ASCEND_CANN_PACKAGE_PATH}/lib64
)

target_link_libraries(minimal_demo PRIVATE
    tiling_api
    register
    platform
    ascendc_runtime
    runtime
    ascendcl
    error_manager
    profapi
    ge_common_base
    unified_dlog
    mmpa
    ascend_dump
    c_sec
    m
    dl
)

target_compile_options(minimal_demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-3510>
)

---

## 6. 构建脚本 build.sh

除了直接在 Jupyter 中运行 CMake 命令，`src/conv2d_op/build.sh` 提供了一个等价的一键构建脚本，适合在终端中使用。

```bash
#!/bin/bash
set -e

SCRIPT_DIR=$(cd "$(dirname "$0")" && pwd)
BUILD_DIR=${SCRIPT_DIR}/build
# TODO: Set ASCEND_HOME_PATH in your environment, or replace the fallback path below
#       with your actual CANN installation directory, e.g. /usr/local/Ascend/ascend-toolkit/latest
CANN_PATH=${ASCEND_HOME_PATH:-/path/to/your/cann}

rm -rf ${BUILD_DIR}
mkdir -p ${BUILD_DIR} && cd ${BUILD_DIR}

cmake .. -DASCEND_CANN_PACKAGE_PATH=${CANN_PATH}
make -j$(nproc)

echo "Build success: ${BUILD_DIR}/minimal_demo"
```

**使用方式**：

```bash
# 步骤一：设置环境变量
export ASCEND_HOME_PATH=/your/cann/install/path
bash src/conv2d_op/build.sh

# 步骤二：修改脚本中的 CANN_PATH 默认值为你的实际路径
```

> **注意**：脚本中 `CANN_PATH` 的默认值 `/path/to/your/cann` 是占位符，请替换为你的实际 CANN 安装目录，或在运行前设置 `ASCEND_HOME_PATH` 环境变量。

---

## 7. 编译与运行

执行以下命令编译可执行文件：

In [ ]:
!cd Sources/01.03 && mkdir -p build
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && \
cd Sources/01.03/build/ && \
cmake .. && \
make

再执行以下代码，进行算子的实际运行：

In [ ]:
!cd Sources/01.03/build/ && ./minimal_demo

---

## 课后实践

本节共有 **9 处填空**，分布在 `SetTilingData`（4 处）和 `CheckTilingSpace`（5 处）中。

请根据注释中的提示，在下面的代码框中补充完整的 `SetTilingData` 和 `CheckTilingSpace` 函数实现。

**SetTilingData 填空（4 处）：**

- `[QUIZ 1/4]` kAL1：L1 侧特征图的 K 维度大小 = KH × KW × CI
- `[QUIZ 2/4]` kBL1：L1 侧权重的 K 维度大小（与 kAL1 相等）
- `[QUIZ 3/4]` kL0：L0 侧每次 Mmad 的 K 维度大小（本 Demo 中等于 kAL1）
- `[QUIZ 4/4]` mStep：M 维度对齐到 16 的倍数，M = HO × WO

**CheckTilingSpace 填空（5 处）：**

- `[QUIZ 1/5]` dilatedKH/W：膨胀后的有效卷积核尺寸
- `[QUIZ 2/5]` hiLoad/wiLoad：L1 需要加载的输入 patch 尺寸
- `[QUIZ 3/5]` al1Bytes：AL1 缓冲区字节数
- `[QUIZ 4/5]` al0Bytes：AL0 缓冲区字节数（im2col 结果）
- `[QUIZ 5/5]` cl0Bytes：L0C 累加器字节数（注意是 fp32！）

In [ ]:
# 在此处填写你的答案，然后运行本单元格验证
# 参考 SetTilingData 和 CheckTilingSpace 中的 [QUIZ] 注释

# [QUIZ 1/4] kAL1 = ?
kAL1 = None  # TODO: KH * KW * CI

# [QUIZ 2/4] kBL1 = ?
kBL1 = None  # TODO: KH * KW * CI

# [QUIZ 3/4] kL0 = ?
kL0 = None   # TODO: KH * KW * CI

# [QUIZ 4/4] mStep = ?
HO = 6; WO = 6
mStep = None  # TODO: AlignUp(HO * WO, 16)

# [QUIZ 1/5] dilatedKH, dilatedKW = ?
KH = 3; KW = 3; DILATION_H = 1; DILATION_W = 1
dilatedKH = None  # TODO: 1 + (KH - 1) * DILATION_H
dilatedKW = None  # TODO: 1 + (KW - 1) * DILATION_W

# [QUIZ 2/5] hiLoad, wiLoad = ?
STRIDE_H = 1; STRIDE_W = 1
hiLoad = None  # TODO: (HO - 1) * STRIDE_H + dilatedKH
wiLoad = None  # TODO: (WO - 1) * STRIDE_W + dilatedKW

# [QUIZ 3/5] al1Bytes = ?
CI = 16
al1Bytes = None  # TODO: hiLoad * wiLoad * CI * 2  (fp16 = 2 bytes)

# [QUIZ 4/5] mAlign, al0Bytes = ?
mAlign = None   # TODO: ((HO * WO + 15) // 16) * 16
al0Bytes = None  # TODO: mAlign * kL0 * 2  (fp16 = 2 bytes)

# [QUIZ 5/5] cl0Bytes = ?
CO = 16
nAlign = ((CO + 15) // 16) * 16
cl0Bytes = None  # TODO: mAlign * nAlign * 4  (fp32 = 4 bytes, NOT 2!)

print("Your answers:")
print(f"  kAL1={kAL1}, kBL1={kBL1}, kL0={kL0}, mStep={mStep}")
print(f"  dilatedKH={dilatedKH}, dilatedKW={dilatedKW}")
print(f"  hiLoad={hiLoad}, wiLoad={wiLoad}")
print(f"  al1Bytes={al1Bytes}, al0Bytes={al0Bytes}, cl0Bytes={cl0Bytes}")

完成填写后，执行以下命令查看答案：

In [ ]:
!cat ./answer/01.03_answer.txt